# Notebook 03: Analise do Mercado de Trabalho — RAIS e CAGED

**Projeto:** `migracao-venezuelana-oeste-sc`  
**Autores:** Leonardo Rafael Santos Leitao e Vicente Neves da Silva Ribeiro (UFFS)  
**Data:** Abril de 2026

---

## Objetivo

Este notebook analisa a insercao da populacao venezuelana no mercado de trabalho formal do Oeste de Santa Catarina.

> **Nota metodologica:** Dados sinteticos utilizados como placeholder.

## 1. Configuracao do Ambiente

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ANOS = list(range(2018, 2024))
print("Ambiente configurado.")

## 2. Dados Sinteticos — RAIS

In [ ]:
np.random.seed(42)
municipios = ["Chapeco", "Xanxere", "Concordia", "Joacaba", "Sao Miguel do Oeste"]
cnaes = {
    "1011-2/01": "Abate de bovinos",
    "1012-1/03": "Abate de suinos",
    "1013-9/01": "Fabricacao de produtos de carne",
    "4120-4/00": "Incorporacao de empreendimentos",
    "4782-2/00": "Varejo de bebidas",
    "5611-2/01": "Restaurantes",
    "9609-2/05": "Servicos de higiene e beleza",
}
records = []
for ano in ANOS:
    n_ven = int(200 + 80 * (ano - 2018))
    n_bra = int(5000 + 100 * (ano - 2018))
    for n, nacionalidade in [(n_ven, "Venezuelana"), (n_bra, "Brasileira")]:
        for _ in range(n):
            cnae_code = np.random.choice(list(cnaes.keys()))
            base_salario = 1800 if nacionalidade == "Venezuelana" else 2800
            salario = max(1212, np.random.lognormal(mean=np.log(base_salario), sigma=0.4))
            records.append({
                "ano": ano,
                "municipio": np.random.choice(municipios),
                "nacionalidade": nacionalidade,
                "cnae": cnae_code,
                "setor": cnaes[cnae_code],
                "idade": int(np.random.normal(32 if nacionalidade == "Venezuelana" else 38, 10)),
                "sexo": np.random.choice([1, 2], p=[0.65, 0.35] if nacionalidade == "Venezuelana" else [0.52, 0.48]),
                "salario_dezembro": round(salario, 2),
                "tempo_emprego_meses": max(1, int(np.random.exponential(18))),
            })
df_rais = pd.DataFrame(records)
print(f"RAIS: {df_rais.shape[0]} registros")
display(df_rais.head())

## 3. Estrutura Setorial — CNAE

In [ ]:
setorial = df_rais.groupby(["setor", "nacionalidade"]).size().reset_index(name="vinculos")
setorial["pct"] = setorial.groupby("nacionalidade")["vinculos"].transform(lambda x: x / x.sum() * 100)

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=setorial, x="pct", y="setor", hue="nacionalidade",
            palette={"Venezuelana": "#d62728", "Brasileira": "#1f77b4"}, ax=ax)
ax.set_title("Distribuicao Setorial (CNAE) — Vinculos Formais (Placeholder)", fontsize=14, fontweight="bold")
ax.set_xlabel("Percentual dos vinculos (%)")
ax.set_ylabel("Setor de atividade")
ax.legend(title="Nacionalidade")
sns.despine()
plt.tight_layout()
fig_path = FIGURES_DIR / "03_rais_estrutura_setorial.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figura salva: {fig_path}")
plt.show()

## 4. Analise Salarial

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=df_rais, x="nacionalidade", y="salario_dezembro",
            palette={"Venezuelana": "#d62728", "Brasileira": "#1f77b4"},
            ax=axes[0], showfliers=False)
axes[0].set_title("Distribuicao Salarial — Dezembro (Placeholder)", fontweight="bold")
axes[0].set_ylabel("Salario (R$)")
sns.despine(ax=axes[0])

salario_setor = df_rais.groupby(["setor", "nacionalidade"])["salario_dezembro"].median().reset_index()
sns.barplot(data=salario_setor, x="salario_dezembro", y="setor", hue="nacionalidade",
            palette={"Venezuelana": "#d62728", "Brasileira": "#1f77b4"}, ax=axes[1])
axes[1].set_title("Salario Mediano por Setor (Placeholder)", fontweight="bold")
axes[1].set_xlabel("Salario mediano (R$)")
sns.despine(ax=axes[1])

plt.tight_layout()
fig_path = FIGURES_DIR / "03_rais_analise_salarial.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figura salva: {fig_path}")
plt.show()

## 5. Dados Sinteticos — CAGED (Rotatividade)

In [ ]:
records_caged = []
for ano in range(2018, 2025):
    for mes in range(1, 13):
        for mun in municipios:
            base = 1.0 + 0.1 * (ano - 2018)
            adm_ven = max(0, int(np.random.poisson(5 * base)))
            des_ven = max(0, int(np.random.poisson(4 * base)))
            adm_bra = max(0, int(np.random.poisson(120 * base)))
            des_bra = max(0, int(np.random.poisson(100 * base)))
            for _ in range(adm_ven):
                records_caged.append({"ano": ano, "mes": mes, "municipio": mun,
                                      "nacionalidade": "Venezuelana", "tipo": "Admissao"})
            for _ in range(des_ven):
                records_caged.append({"ano": ano, "mes": mes, "municipio": mun,
                                      "nacionalidade": "Venezuelana", "tipo": "Desligamento"})
            for _ in range(adm_bra):
                records_caged.append({"ano": ano, "mes": mes, "municipio": mun,
                                      "nacionalidade": "Brasileira", "tipo": "Admissao"})
            for _ in range(des_bra):
                records_caged.append({"ano": ano, "mes": mes, "municipio": mun,
                                      "nacionalidade": "Brasileira", "tipo": "Desligamento"})
df_caged = pd.DataFrame(records_caged)
print(f"CAGED: {df_caged.shape[0]} movimentacoes")

## 6. Saldo de Emprego e Rotatividade

In [ ]:
caged_agg = df_caged.groupby(["ano", "nacionalidade", "tipo"]).size().reset_index(name="quantidade")
caged_pivot = caged_agg.pivot(index=["ano", "nacionalidade"], columns="tipo", values="quantidade").fillna(0).reset_index()
caged_pivot["saldo"] = caged_pivot["Admissao"] - caged_pivot["Desligamento"]
caged_pivot["turnover"] = ((caged_pivot["Admissao"] + caged_pivot["Desligamento"]) / 2) / \n                            (caged_pivot["Admissao"] - caged_pivot["Desligamento"].cumsum() + 1000) * 100

fig, ax = plt.subplots(figsize=(14, 6))
for nat, color in [("Venezuelana", "#d62728"), ("Brasileira", "#1f77b4")]:
    sub = caged_pivot[caged_pivot["nacionalidade"] == nat]
    ax.plot(sub["ano"], sub["saldo"], marker="o", label=nat, color=color, linewidth=2)
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Saldo de Emprego — CAGED (Placeholder)", fontsize=14, fontweight="bold")
ax.set_xlabel("Ano")
ax.set_ylabel("Saldo (Admissoes - Desligamentos)")
ax.legend(title="Nacionalidade")
sns.despine()
plt.tight_layout()
fig_path = FIGURES_DIR / "03_caged_saldo_emprego.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figura salva: {fig_path}")
plt.show()

## 7. Salvamento dos Dados

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
df_rais.to_csv(DATA_DIR / "gold" / "rais_vinculos.csv", index=False)
df_caged.to_csv(DATA_DIR / "gold" / "caged_movimentacoes.csv", index=False)
print("Dados exportados com sucesso.")

## Proximos Passos

- [ ] Obter bases reais RAIS/CAGED do Ministerio do Trabalho;
- [ ] Implementar analise de sobrevivencia (Kaplan-Meier) com lifelines;
- [ ] Cruzar com CNAE do setor frigorifico para analise especifica;
- [ ] Calcular indices de Gini e Theil para desigualdade salarial.